# 🚀 ScreenerPro AI: Resume Screening and Skill Matching System
This notebook demonstrates the end-to-end pipeline in a Google Colab environment.

### 📋 Steps to Run:
1. Run the **Environment Setup** cell to clone the repository and install dependencies.
2. Run the **Pipeline** cells to see the candidate ranking and metrics.

In [ ]:
# --- 1. Environment Setup ---
!git clone https://github.com/defender07/claysis.git
%cd claysis/resume-screening-system

# Install dependencies
!pip install -r requirements.txt
!python -m spacy download en_core_web_sm

In [ ]:
# --- 2. Imports ---
import os
import sys
import time
import numpy as np
from src.ingestion import load_documents_from_directory
from src.preprocessing import preprocess, extract_skills
from src.embedding import generate_embedding, generate_embeddings_batch
from src.ranking import rank_candidates, evaluate_metrics
print("Setup complete!")

In [ ]:
# --- 3. Run Analysis Pipeline ---
resumes_dir = 'data/resumes'
jd_dir = 'data/job_descriptions'

resumes = load_documents_from_directory(resumes_dir)
jds = load_documents_from_directory(jd_dir)
jd_filename, jd_text = list(jds.items())[0]
resume_filenames = list(resumes.keys())

print(f"Analyzing {len(resumes)} resumes against {jd_filename}...\n")

# Perform Ranking
start_time = time.time()
ranked_results = rank_candidates(jd_text, list(resumes.values()), resume_filenames)
processing_time = time.time() - start_time

print("🎯 Ranked Candidates:")
for rank, res in enumerate(ranked_results, 1):
    suitability = "✅ Suitable" if res["is_suitable"] else "❌ Not Suitable"
    print(f"{rank}. {res['filename']} [{suitability}] - Score: {res['score']:.4f}")
    print(f"   Explanation: {res['explanation']}")
    print(f"   Matched Skills: {res['matched_skills']}\n")

print(f"Processing Time: {processing_time:.4f} seconds")

In [ ]:
# --- 4. Evaluation Metrics ---
# Ground truth for jd_python.txt in the sample data
ground_truth = {'resume_a.txt'}
ranked_filenames = [res["filename"] for res in ranked_results]
metrics = evaluate_metrics(ranked_filenames, ground_truth)

print("📈 Evaluation Metrics:")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")